# Preprocessing the PICU data modified by Derek

In [1]:
# NEEDED:
#!pip install mne==1.2
#!pip install autoreject
# Note: need file structure s.t. 
# preprocessing {Base}
    # outputs
        # epochs
        # raw_filtered
        # reports

# To add:
## set up to select specific subjects
## add BIDS Sub layout for each case. 
## Add os.makedir if DNE

In [ ]:
"""
The script uses the Anaconda environment : mne1.2
"""
import mne
import os
import matplotlib.pyplot as plt
from mne.preprocessing import ICA
from autoreject import (AutoReject, set_matplotlib_defaults)
import numpy as np
from pathlib import Path

import traceback # Derek addition

def prepare_raw(raw, report):
    """
    Specify electrode type (for ICA), Set montage
    """
    # Labeling EOG, EKG, and EMG channels as such.
    raw = raw.copy().set_channel_types(mapping = {'LEOG': 'eog', 'REOG': 'eog', 'X1': 'eog', 'X4': 'eog',
                                                  'LEKG': 'ecg', 'REKG': 'ecg', 'LEMG': 'emg', 
                                                  'REMG': 'emg', 'Remg': 'emg', 'Lemg': 'emg'})
    # Drop the OSAT and PR channels
    raw = raw.copy().drop_channels(['OSAT', 'PR', 'DC1', 'DC2', 'DC3', 'DC4'])
        
    # Get number of channels  
    print('Number of channels:')
    print(len(raw.ch_names))

    # Set montage
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore') # Ignoring 18 channels not in montage (!!!)
    fig = raw.plot_sensors(show_names=True)
    report.add_figure(
        fig=fig, title='Plot sensors of montage standard_1020',
        image_format='PNG'
    )
    plt.close('all')

    report.add_raw(raw=raw, title="Raw", psd=True)
    
    return(raw, report)
 
def filter_raw(raw, report):
    """
    Filter the raw data and plot the PSD
    """
    # Filter the data between 0.1 to 50 Hz
    eeg_filtered = raw.filter(l_freq=0.5, h_freq = 45)

    # Notch filter the data for freq = 60Hz and 120Hz to remove electrical noise
    eeg_notch = eeg_filtered.copy().notch_filter(freqs=[60, 120])

    report.add_raw(raw=eeg_notch, title="Raw filtered", psd=True)
    
    return(eeg_notch, report)

def reref(raw, report):
    """
    Re-reference the data to average of mastoids.
    """
    eeg_new_ref = raw.set_eeg_reference(ref_channels = ['A1', 'A2']) 

    report.add_raw(raw=eeg_new_ref, title="Raw re-referenced", psd=True)
    
    return(eeg_new_ref, report)

def ica_raw(raw, report, filename):
    ica = ICA(n_components=0.99, max_iter='auto', random_state=42)
    ica.fit(raw)
    fig = ica.plot_sources(raw)
    report.add_figure(
        fig=fig, title='ICA sources',
        image_format='PNG'
    )
    plt.close()

    ica.exclude = []
    # find which ICs match the EOG pattern
    eog_epochs = mne.preprocessing.create_eog_epochs(raw=raw)
    eog_indices, eog_scores = ica.find_bads_eog(raw, 
                                                # measure='zscore', 
                                                # threshold=2
                                                ) 
    ecg_epochs = mne.preprocessing.create_ecg_epochs(raw=raw)
    ecg_indices, ecg_scores = ica.find_bads_ecg(raw, 
                                                # method='correlation',
                                                threshold='auto')
    
    # Check which components we will remove
    ica.exclude.extend(ecg_indices)
    ica.exclude.extend(eog_indices)

    ## Remove components
    eeg_postica = ica.apply(raw)

    report.add_ica(
        ica=ica,
        title="ICA EOG & ECG",
        inst=raw,
        eog_evoked=eog_epochs.average(),
        ecg_evoked=ecg_epochs.average(),
        eog_scores=eog_scores,
        ecg_scores=ecg_scores,
        n_jobs=None,
    )    

    fig = eeg_postica.compute_psd(picks='eeg', fmax = 70).plot(average=False)
    plt.title(filename)
    report.add_figure(
        fig=fig, title='PSD post ICA',
        image_format='PNG'
    )
    plt.close()
    
    return(eeg_postica, report)

def epoching(raw, report):
    """
    Epoch the data
    """
    epoch_duration = 10 
    new_events = mne.make_fixed_length_events(raw, duration=epoch_duration, overlap=0)
    report.add_events(events=new_events, 
                      title=f'Generated Events {epoch_duration} sec apart', 
                      sfreq=raw.info['sfreq'])
    # Decimation params
    desired_sfreq = 250  # Hz
    current_sfreq = raw.info['sfreq'] # Hz
    decim = np.round(current_sfreq / desired_sfreq).astype(int)
    print(f'Decimating the data to : {desired_sfreq} Hz, with a decim of : {decim}.')
    my_html = f""" <p> The sampling frequency of the data is : {current_sfreq} Hz.<br>
    Decimating the data to : {desired_sfreq} Hz, with a decim of : {decim}.<br> </p>"""
    report.add_html(title='Decimation', html=my_html)

    epochs = mne.Epochs(raw, new_events, 
                        baseline=None, 
                        decim=decim,
                        tmin=0, tmax=epoch_duration) # to epoch_duration set to 10
    return(epochs, report)

def apply_autoreject_cleaning(epochs, report, filename):
    epochs.load_data()
    
    # Autoreject params 
    n_interpolates = np.array([1, 2])
    consensus_percs = np.linspace(0, 1.0, 11)

    # Plot and save the PSD before the cleaning
    fig = epochs.compute_psd(picks='eeg', fmax = 70).plot(average=False)
    plt.title(filename)
    report.add_figure(
        fig=fig, title='PSD pre-Autoreject', image_format='PNG'
    )
    plt.close()

    # Run Autoreject
    ar = AutoReject(n_interpolates, consensus_percs,
                thresh_method='random_search', random_state=42)
    ar.fit(epochs)
    epochs_clean, reject_log = ar.transform(epochs, return_log=True)
    #### Files fail when no epochs are removed: Added by Derek
    if reject_log.bad_epochs.any():
        # Plot the signals of the rejected epochs
        fig = epochs[reject_log.bad_epochs].plot(scalings=dict(eeg=100e-6))
        report.add_figure(fig=fig, title='Autoreject bad epochs',image_format='PNG')
        plt.close()

        # Plot the reject log in a carpet plot
        fig, axes = plt.subplots(figsize=(5, 40))
        fig2 = reject_log.plot(orientation='vertical', ax=axes, show_names=1)
        report.add_figure(fig=fig2, title='Autoreject bad sensors heatmap',image_format='PNG')
        plt.close(fig)
        plt.close(fig2)
        plt.close('all')
    else:
        print("No bad epochs to plot.")
    # Plot and save the PSD after the cleaning
    fig = epochs_clean.compute_psd(picks='eeg', fmax = 70).plot(average=False)
    plt.title(filename)
    report.add_figure(
        fig=fig, title='PSD post-Autoreject',
        image_format='PNG'
    )
    plt.close()

    return(epochs_clean, report)

In [ ]:
# Run preprocessing
hospital = 'Pediatric_Complexity' 
base_path = f'D:/Derek/{hospital}/preprocessing/'
location = f"{base_path}Pediatric Anesthesia EEG Raw/Functional Data"
participant_ID = []

# Iterate over patient folders
with os.scandir(location) as entries:
    for entry in entries:
        participant_ID.append(entry.name)
        print(f'{entry.name}')

In [ ]:
signal_threshold = 500*1e-6
failed_files = []
skipped_files = []
files_list = []
# participant_ID: Get a patient ID
for partic in participant_ID:
    ID = partic
    print(f'Patient ID : {ID}')
    tasksEEG = []
    
    with os.scandir(location + '/' + ID) as files:
        for file in files:                
            # If the report exists already, skip the file
            if os.path.exists(f"{base_path}/outputs/reports/{file.name.split('.')[0]}_report.html"):
                print(f'Skipping {file.name}')
                skipped_files.append(file.name)
                continue
            try: 
                print(file.name)
                files_list.append(file.name)
            except Exception as e:
                failed_files.append(file.name)
                print(f'###### Watch out! Failed file : {file.name}')
                print(f"     With Error: {e}")
                traceback.print_exc()  
print(f'The following files were not processed : {failed_files}')
len(files_list)

In [ ]:
signal_threshold = 500*1e-6
failed_files = []
skipped_files = []
errors_list = []

# participant_ID: Get a patient ID
for partic in participant_ID:
    ID = partic
    print(f'Patient ID : {ID}')
    tasksEEG = []
    
    with os.scandir(location + '/' + ID) as files:
        for file in files:                
            # If the report exists already, skip the file
            if os.path.exists(f"{base_path}/outputs/reports/{file.name.split('.')[0]}_report.html"):
                print(f'Skipping {file.name}')
                skipped_files.append(file.name)
                continue
            try: 
                tasksEEG.append(file)
                filename = file.name.split('.')[0]
                print(f'#################################### {filename} ####################################')
                report = mne.Report(title=filename)
                raw = mne.io.read_raw_edf(os.path.join(location, partic, file.name), preload=True)

                raw, report = prepare_raw(raw, report)
                raw, report = filter_raw(raw, report)
                raw, report = reref(raw, report)
                raw = raw.drop_channels(['T1', 'T2'])
                raw, report = ica_raw(raw, report, filename)
                raw.save(f'{base_path}/outputs/raw_filtered/{filename}_ICA_filtered_raw.fif', overwrite=True) # change to noICA if remove
                print(f'{ID} file saved to: {base_path}/outputs/raw_filtered/{filename}_ICA_filtered_raw.fif') # same as above 
                epochs, report = epoching(raw, report)
                report.add_epochs(epochs=epochs, title='Epochs before Autoreject')
                epochs.load_data()
                epochs.drop_bad({'eeg':signal_threshold})
                fig = epochs.plot_drop_log() 
                report.add_figure(
                    fig=fig, title=f'Epochs drop log : {signal_threshold}',
                    image_format='PNG'
                )
                plt.close('all')  

                # Remove all the non-EEG channels and the A1 and A2
                epochs = epochs.drop_channels(['A1', 'A2'])
                epochs = epochs.pick_types(eeg=True)
                epochs, report = apply_autoreject_cleaning(epochs, report, filename)
                report.add_epochs(epochs=epochs, title='Epochs after Autoreject')
                
                mne_init_py_path = Path('D:/Derek/McMaster/scripts/pediatric_proc-Dragana/01_preprocessing.py')
                report.add_code(code=mne_init_py_path,
                    title="Code used to preprocess the data and generate this report.")

                my_html = """ <p>The preprocessing pipeline was developed by Kira Dolhan, adapted by Dragana Manasova and modified by Derek Newman, under the supervision of Dr. Stefanie Blain-Moraes.<br>
                The pipeline is used to preprocess pediatric data. <br> This work is done at the Biosignal Interaction and Personhood Technology (BIAPT) Lab, McGill University, Montreal, Canada. </p>
                """
                report.add_html(title='Acknowledgements', html=my_html)
 
                epochs.save(f'{base_path}/outputs/epochs/{filename}_epochs.fif', overwrite=True)
                print(f'{ID} epoch saved to: {base_path}/outputs/epochs/{filename}_ICA_filtered_raw.fif')

                report.save(f"{base_path}/outputs/reports/{filename}_report.html", overwrite=True)
                print(f'{ID} report saved to: {base_path}/outputs/raw_filtered/{filename}_report.html')
            except Exception as e:
                failed_files.append(file.name)
                print(f'###### Watch out! Failed file : {file.name}')
                print(f"     With Error: {e}")
                errors_list.append(e)
                traceback.print_exc()  
print(f'The following files were not processed : {failed_files}')
print(f'The following files were not processed because: {errors_list}')
